In [4]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── 1. Find optimal n_comp per target separately ──────────────────────────────
print("="*60)
print("FINDING OPTIMAL n_comp PER TARGET")
print(f"{'n_comp':<10} {'Mem r':>8} {'Brand r':>8}")
print("-"*30)

best_mem_score,   best_mem_ncomp   = -99, 50
best_brand_score, best_brand_ncomp = -99, 50

df = pd.read_csv('devset_videolist_GT.csv')
y_mem   = df['memorability_score'].values
y_brand = df['brand_memorability'].values




# ── Brand-Isolated Cross-Validation Folds ─────────────────────────────────
# Goldman Sachs (79 videos) gets its own fold due to size
# Other channels grouped into ~4 balanced folds

channel_counts = df['channelName'].value_counts()
print("Building brand-isolated folds...")
print(channel_counts.to_string())

# Assign each channel to a fold
# Fold 0: Goldman Sachs alone (79 videos, 23%)
# Folds 1-4: remaining channels grouped by size (greedy bin packing)

N_FOLDS = 5
fold_assignments = {}  # channel -> fold_id

# Goldman Sachs gets fold 0
fold_assignments['Goldman Sachs'] = 0
fold_sizes = {0: 79, 1: 0, 2: 0, 3: 0, 4: 0}

# Assign remaining channels greedily to smallest fold (1-4)
remaining = channel_counts.drop('Goldman Sachs')
for channel, count in remaining.items():
    # Pick fold with smallest current size among folds 1-4
    target_fold = min([1, 2, 3, 4], key=lambda f: fold_sizes[f])
    fold_assignments[channel] = target_fold
    fold_sizes[target_fold] += count

# Map each video to its fold
df['fold'] = df['channelName'].map(fold_assignments)
print("\nFold sizes (videos):")
print(df['fold'].value_counts().sort_index())
print("\nChannels per fold:")
for f in range(N_FOLDS):
    channels = [ch for ch, fold in fold_assignments.items() if fold == f]
    print(f"  Fold {f}: {channels}")

# ── Load all transcripts ───────────────────────────────────────────────────
transcripts = {}
missing_stt = []
STT_DIR = Path('devset-stt')
for vid_id in df['id']:
    found = False
    for name in [vid_id, f'_{vid_id}']:
        path = STT_DIR / f'{name}.txt'
        if path.exists():
            transcripts[vid_id] = path.read_text(encoding='utf-8').strip()
            found = True
            break
    if not found:
        transcripts[vid_id] = ''
        missing_stt.append(vid_id)

print(f"Missing STT files: {len(missing_stt)}")
print(f"Empty transcripts: {sum(1 for t in transcripts.values() if t == '')}")

corpus = [transcripts[vid_id] for vid_id in df['id']]
# TF-IDF on full corpus but with better params
tfidf_v2 = TfidfVectorizer(
    max_features=3000,
    min_df=3,
    max_df=0.80,
    ngram_range=(1, 3),   # include trigrams
    sublinear_tf=True,
    strip_accents='unicode',
    stop_words='english',
    analyzer='word'
)
X_tfidf_v2   = tfidf_v2.fit_transform(corpus)


def rank_aggregate_cv(feature_dict, y_mem, y_brand, df):
    """
    For each fold:
      - Compute Spearman r of each feature vs target ON TRAIN SET
      - Sign-flip features that are negatively correlated
      - Average the ranks across features
      - Evaluate on val set
    No model fitting — just rank averaging.
    """
    preds_mem   = np.zeros(len(df))
    preds_brand = np.zeros(len(df))
    feat_names  = list(feature_dict.keys())
    X_all       = np.column_stack(list(feature_dict.values()))

    for fold_id in range(5):
        val_mask   = (df['fold'] == fold_id).values
        train_mask = ~val_mask

        X_tr  = X_all[train_mask]
        X_val = X_all[val_mask]
        n_val = val_mask.sum()

        # Compute direction on train set for each target
        for y, preds in [(y_mem, preds_mem), (y_brand, preds_brand)]:
            y_tr = y[train_mask]
            directions = []
            valid_cols  = []
            for j in range(X_tr.shape[1]):
                col = X_tr[:, j]
                if len(np.unique(col)) < 2:
                    continue
                r, _ = spearmanr(col, y_tr)
                if abs(r) < 0.03:   # skip near-zero features this fold
                    continue
                directions.append(np.sign(r))
                valid_cols.append(j)

            if not valid_cols:
                preds[val_mask] = y_tr.mean()
                continue

            # Rank each feature on val set, flip direction, average
            agg = np.zeros(n_val)
            for j, sign in zip(valid_cols, directions):
                col_val = X_val[:, j]
                ranked  = rankdata(col_val).astype(float)
                agg    += sign * ranked

            preds[val_mask] = agg  # higher = more memorable

    r_m = spearmanr(y_mem,   preds_mem).correlation
    r_b = spearmanr(y_brand, preds_brand).correlation
    return r_m, r_b, preds_mem, preds_brand



for n_comp in [50, 80, 100, 120, 150, 200]:
    svd_t = TruncatedSVD(n_components=n_comp, random_state=42)
    X_lsa_t = svd_t.fit_transform(X_tfidf_v2)
    feats_t = {f'lsa_{i}': X_lsa_t[:, i] for i in range(n_comp)}
    r_m, r_b, _, _ = rank_aggregate_cv(feats_t, y_mem, y_brand, df)
    print(f"{n_comp:<10} {r_m:>8.4f} {r_b:>8.4f}")
    if r_m > best_mem_score:
        best_mem_score, best_mem_ncomp = r_m, n_comp
    if r_b > best_brand_score:
        best_brand_score, best_brand_ncomp = r_b, n_comp

print(f"\nBest n_comp for memorability:       {best_mem_ncomp} (r={best_mem_score:.4f})")
print(f"Best n_comp for brand memorability: {best_brand_ncomp} (r={best_brand_score:.4f})")


# ── 2. Final cross-val with per-target optimal configs ────────────────────────
print("\n" + "="*60)
print("FINAL CV RESULTS — per-target optimal configs")

# Separate SVDs for each target
svd_mem   = TruncatedSVD(n_components=best_mem_ncomp,   random_state=42)
svd_brand = TruncatedSVD(n_components=best_brand_ncomp, random_state=42)

X_lsa_mem   = svd_mem.fit_transform(X_tfidf_v2)
X_lsa_brand = svd_brand.fit_transform(X_tfidf_v2)

feats_mem   = {f'lsa_{i}': X_lsa_mem[:,   i] for i in range(best_mem_ncomp)}
feats_brand = {f'lsa_{i}': X_lsa_brand[:, i] for i in range(best_brand_ncomp)}

r_m, _, final_preds_mem,   _ = rank_aggregate_cv(feats_mem,   y_mem, y_brand, df)
_, r_b, _, final_preds_brand = rank_aggregate_cv(feats_brand, y_mem, y_brand, df)
print(f"  Memorability Spearman r:       {r_m:.4f}")
print(f"  Brand Memorability Spearman r: {r_b:.4f}")


# ── 3. Per-fold final breakdown ───────────────────────────────────────────────
print("\nPer-fold breakdown:")
print(f"  {'Fold':<6} {'n':>4}  {'Mem r':>7}  {'Brand r':>8}  Channels")
print("  " + "-"*65)

all_pm = np.zeros(len(df))
all_pb = np.zeros(len(df))

for fold_id in range(5):
    val_mask   = (df['fold'] == fold_id).values
    train_mask = ~val_mask

    agg_m = np.zeros(val_mask.sum())
    agg_b = np.zeros(val_mask.sum())

    for j in range(best_mem_ncomp):
        col_tr, col_val = X_lsa_mem[train_mask, j], X_lsa_mem[val_mask, j]
        r, _ = spearmanr(col_tr, y_mem[train_mask])
        if abs(r) > 0.03:
            agg_m += np.sign(r) * rankdata(col_val).astype(float)

    for j in range(best_brand_ncomp):
        col_tr, col_val = X_lsa_brand[train_mask, j], X_lsa_brand[val_mask, j]
        r, _ = spearmanr(col_tr, y_brand[train_mask])
        if abs(r) > 0.03:
            agg_b += np.sign(r) * rankdata(col_val).astype(float)

    all_pm[val_mask] = agg_m
    all_pb[val_mask] = agg_b

    r_m_f = spearmanr(y_mem[val_mask],   agg_m).correlation
    r_b_f = spearmanr(y_brand[val_mask], agg_b).correlation
    chs   = ', '.join(df[val_mask]['channelName'].unique()[:2])
    print(f"  {fold_id:<6} {val_mask.sum():>4}  {r_m_f:>7.4f}  {r_b_f:>8.4f}  {chs}...")


# ── 4. Train FINAL models on ALL 339 videos ───────────────────────────────────
# For test set prediction we need:
# (a) TF-IDF fitted on all train transcripts
# (b) SVD fitted on all train TF-IDF
# (c) Direction of each LSA dim w.r.t. target on all train videos
# Then apply to test transcripts

print("\n" + "="*60)
print("FITTING FINAL MODELS ON ALL 339 TRAINING VIDEOS")

# Refit TF-IDF on all train data (already done as tfidf_v2)
# Refit SVD with optimal n_comp per target
final_svd_mem   = TruncatedSVD(n_components=best_mem_ncomp,   random_state=42)
final_svd_brand = TruncatedSVD(n_components=best_brand_ncomp, random_state=42)

X_train_lsa_mem   = final_svd_mem.fit_transform(X_tfidf_v2)
X_train_lsa_brand = final_svd_brand.fit_transform(X_tfidf_v2)

# Compute direction of each dim on full training set
mem_directions   = []
brand_directions = []
mem_dims_used    = []
brand_dims_used  = []

for j in range(best_mem_ncomp):
    r, _ = spearmanr(X_train_lsa_mem[:, j], y_mem)
    if abs(r) > 0.03:
        mem_directions.append(np.sign(r))
        mem_dims_used.append(j)

for j in range(best_brand_ncomp):
    r, _ = spearmanr(X_train_lsa_brand[:, j], y_brand)
    if abs(r) > 0.03:
        brand_directions.append(np.sign(r))
        brand_dims_used.append(j)

print(f"  Dims used for memorability:       {len(mem_dims_used)}/{best_mem_ncomp}")
print(f"  Dims used for brand memorability: {len(brand_dims_used)}/{best_brand_ncomp}")


# ── 5. Prediction function for test set ───────────────────────────────────────
def predict_test_videos(test_transcripts_dict, test_video_ids):
    """
    Given a dict of {video_id: transcript_text}, returns predictions.
    
    Parameters:
        test_transcripts_dict: dict {video_id: str}
        test_video_ids: list of video IDs in order
    
    Returns:
        pred_mem, pred_brand: arrays of shape (n_test,)
    """
    # Transform test transcripts with TRAIN-fitted TF-IDF
    test_corpus = [test_transcripts_dict.get(vid, '') for vid in test_video_ids]
    X_test_tfidf = tfidf_v2.transform(test_corpus)  # uses train vocab

    # Project into LSA space
    X_test_lsa_mem   = final_svd_mem.transform(X_test_tfidf)
    X_test_lsa_brand = final_svd_brand.transform(X_test_tfidf)

    # Rank aggregation: rank WITHIN test set, apply train-computed directions
    # Note: for a proper submission we rank within the test set
    pred_mem   = np.zeros(len(test_video_ids))
    pred_brand = np.zeros(len(test_video_ids))

    for j, sign in zip(mem_dims_used, mem_directions):
        pred_mem += sign * rankdata(X_test_lsa_mem[:, j]).astype(float)

    for j, sign in zip(brand_dims_used, brand_directions):
        pred_brand += sign * rankdata(X_test_lsa_brand[:, j]).astype(float)

    # Normalize to [0, 1] range to match score distribution
    def normalize(arr):
        mn, mx = arr.min(), arr.max()
        if mx == mn: return np.full_like(arr, 0.5)
        return (arr - mn) / (mx - mn)

    # Scale to match train score range
    pred_mem_norm   = normalize(pred_mem)
    pred_brand_norm = normalize(pred_brand)

    # Map to train score range
    pred_mem_scaled   = (pred_mem_norm *
                         (y_mem.max()   - y_mem.min())   + y_mem.min())
    pred_brand_scaled = (pred_brand_norm *
                         (y_brand.max() - y_brand.min()) + y_brand.min())

    return pred_mem_scaled, pred_brand_scaled


# ── 6. Validate predict_test_videos on train data (sanity check) ─────────────
print("\nSanity check — applying predict_test_videos to train set:")
train_transcript_dict = {vid: transcripts[vid] for vid in df['id']}
pm_check, pb_check = predict_test_videos(train_transcript_dict,
                                          df['id'].tolist())
r_m_check = spearmanr(y_mem,   pm_check).correlation
r_b_check = spearmanr(y_brand, pb_check).correlation
print(f"  Mem r (train, expected ~1.0):   {r_m_check:.4f}")
print(f"  Brand r (train, expected ~1.0): {r_b_check:.4f}")
print("  (High values confirm pipeline works correctly)")


# ── 7. Save everything needed for test prediction ─────────────────────────────
import pickle

pipeline = {
    'tfidf':              tfidf_v2,
    'svd_mem':            final_svd_mem,
    'svd_brand':          final_svd_brand,
    'mem_dims_used':      mem_dims_used,
    'mem_directions':     mem_directions,
    'brand_dims_used':    brand_dims_used,
    'brand_directions':   brand_directions,
    'y_mem_min':          float(y_mem.min()),
    'y_mem_max':          float(y_mem.max()),
    'y_brand_min':        float(y_brand.min()),
    'y_brand_max':        float(y_brand.max()),
    'best_mem_ncomp':     best_mem_ncomp,
    'best_brand_ncomp':   best_brand_ncomp,
    'cv_mem_spearman':    r_m,
    'cv_brand_spearman':  r_b,
}

with open('memorability_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

print("\n" + "="*60)
print("Pipeline saved to memorability_pipeline.pkl")
print("\nFINAL SUMMARY:")
print(f"  CV Memorability Spearman r:       {r_m:.4f}")
print(f"  CV Brand Memorability Spearman r: {r_b:.4f}")
print(f"  Method: LSA({best_mem_ncomp}/{best_brand_ncomp}) + rank aggregation")
print(f"  Features: TF-IDF on STT transcripts only")
print(f"  No model parameters fitted (zero overfitting risk)")
print("\nTo predict test set, load the pipeline and call:")
print("  pred_mem, pred_brand = predict_test_videos(test_dict, test_ids)")

FINDING OPTIMAL n_comp PER TARGET
n_comp        Mem r  Brand r
------------------------------
Building brand-isolated folds...
channelName
Goldman Sachs                                   79
UBS                                             37
jpmorgan                                        31
Aon                                             18
Credit Suisse                                   17
Vanguard                                        17
Allianz                                         14
Legal & General Investment Management - LGIM    14
HSBC UK                                         12
Janus Henderson Investment Trusts               11
Legal & General                                 10
BMO Global Asset Management - EMEA               8
Invesco                                          8
Amundi                                           7
Blackstone                                       7
BNP Paribas                                      6
Aviva                                        